# EEEM068 — Knee Abnormality ViT (Colab Runbook)
Run this notebook from top to bottom for a full Colab training + evaluation + attention workflow.


In [ ]:
GITHUB_REPO      = "https://github.com/sachithgowda-cloud/knee-abnormality-vit.git"
DRIVE_BASE       = "/content/drive/MyDrive/knee-abnormality-vit"
OUTPUT_DIR       = f"{DRIVE_BASE}/outputs"
SIT_WEIGHTS_PATH = f"{DRIVE_BASE}/weights/sit-s.pth"
REPO_DIR         = "/content/knee-abnormality-vit"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone {GITHUB_REPO} {REPO_DIR}

%cd /content/knee-abnormality-vit
!pip install -q -r requirements.txt

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Full Training
Run the full training job. If the SiT checkpoint is not present in Drive, switch to the fallback cell below.


In [ ]:
%cd /content/knee-abnormality-vit
!python3 scripts/train.py --colab --sit-weights {SIT_WEIGHTS_PATH}


### Fallback Training Without SiT Weights
Only run this cell if the cell above fails because the SiT checkpoint is missing.


In [ ]:
%cd /content/knee-abnormality-vit
!python3 scripts/train.py --colab


## Post-Training Steps
After training finishes, copy the run folder printed by the training script into `RUN_DIR` below, then run the evaluation and attention cells.


In [ ]:
RUN_DIR = f"{OUTPUT_DIR}/run_YYYYMMDD_HHMMSS"
print(RUN_DIR)


In [ ]:
!python3 scripts/evaluate.py --colab --checkpoint {RUN_DIR}/best_model.pth


In [ ]:
!python3 scripts/visualize_attention.py --colab --checkpoint {RUN_DIR}/best_model.pth --selection correct --num-samples 12 --output-dir {RUN_DIR}/attention_correct


In [ ]:
!python3 scripts/visualize_attention.py --colab --checkpoint {RUN_DIR}/best_model.pth --selection incorrect --num-samples 12 --output-dir {RUN_DIR}/attention_incorrect


In [ ]:
!python3 scripts/review_attention.py --attention-dir {RUN_DIR}/attention_correct
